In [2]:
!pip install -U langchain langchain_classic langchain_huggingface langchain_community groq langchain-groq faiss-cpu tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 1.5 MB/s eta 0:00:00
  Attempting uninstall: groq
    Found existing installation: groq 1.4.0
    Uninstalling groq-1.4.0:
      Successfully uninstalled groq-1.4.0


In [4]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

In [5]:
# creating the docuemnts
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [7]:
# creating the embedding
embedding_model = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
# creating the vector store
vector_store = FAISS.from_documents(docs, embedding_model)

In [10]:
#in the contexual compression yu have to give 2 things baseretriever and base compressor
# creating a base retriever
base_retriever = vector_store.as_retriever(search_kwargs={"k":5})

In [11]:
# creating a base compressor
import os
os.environ["GROQ_API_KEY"] ="your api key"

model = ChatGroq(model = "openai/gpt-oss-120b")

compressor = LLMChainExtractor.from_llm(model)

In [13]:
# creating out contexual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever = base_retriever,
    base_compressor=compressor
)

In [14]:
# suppose we have a query
query = "what is photosynthesis"
compressed_result = compression_retriever.invoke(query)

In [15]:
# printing the results
for i, doc in enumerate(compressed_result):
  print(f"----result {i+1}----")
  print(doc.page_content)

----result 1----
Photosynthesis is the process by which green plants convert sunlight into energy.
----result 2----
The chlorophyll in plant cells captures sunlight during photosynthesis.
----result 3----
Photosynthesis does not occur in animal cells.
